# Limitations of Orbital Uncertainty Propagation Methods

Orbital uncertainty propagation methods differ mainly in how they balance three competing objectives: computational efficiency, fidelity to nonlinear dynamics, and the ability to represent non-Gaussian probability distributions. No single method is optimal in all regimes. Methods that are fast are usually based on local linearity or low-order statistics, while methods that can represent strongly distorted probability densities are usually more expensive or more difficult to scale.

# Summary Table

| Method / framework | Principal strength | Main limitation | Failure mechanism / critical issue | Why the limitation matters | Dynamics Intrusiveness | Open source Implementation | Result Accuracy and Computational cost |
|-------|-----------|-----------|--------------|--------------------------|--|--|--|
| **Monte Carlo (MC)** | Asymptotically exact and distribution-agnostic | Computationally expensive and slow to converge | Poor convergence in rare-event regimes and long propagation horizons | It provides the most direct baseline, but the number of samples required for accurate statistics becomes very large, especially for long propagation times and rare-event probabilities. | TBD | TBD | TBD |
| **Linearised covariance (LinCov / STM)** | Extremely efficient first-order propagation | Relies on local linearity and a Gaussian description of uncertainty | Breakdown of linear approximation under nonlinear orbital dynamics (e.g., stretching, folding) | As orbital dynamics become nonlinear, the propagated distribution departs from an ellipse and LinCov can no longer represent the true uncertainty accurately. | TBD | TBD | TBD |
| **Unscented Transform (UT)** | Captures nonlinear effects up to second-order moments with few samples | Captures only low-order moments and depends on a small sigma-point set | Moment closure failure for skewed or multimodal distributions | UT improves on linearisation, but it still cannot fully describe strongly skewed, multimodal, or highly stretched distributions. | TBD | TBD | TBD |
| **Polynomial Chaos Expansion (PCE)** | Spectral accuracy for smooth stochastic systems | Suffers from the curse of dimensionality | Exponential growth of basis size with dimension/order | The number of basis terms grows rapidly with dimension and polynomial order, which makes the method difficult to scale to realistic orbital problems. | TBD | TBD | TBD |
| **Non-intrusive PCE (NIPC)** | Black-box compatibility with existing propagators | Needs many model evaluations and struggles with low smoothness | Regression instability and sampling inefficiency in high dimensions | It is attractive because it treats the dynamics as a black box, but it becomes costly in high dimension and less reliable when the response surface is not smooth. | TBD | TBD | TBD |
| **Intrusive PCE** | High efficiency once governing equations are reformulated | Requires reformulation of the governing equations | Structural rigidity and implementation complexity of stochastic Galerkin system | The Galerkin system must be derived and solved directly, which increases implementation effort and makes the method less flexible for complex force models. | TBD | TBD | TBD |
| **State Transition Tensors (STT)** | High-order nonlinear state mapping accuracy | High derivational and computational complexity | Tensor order explosion with state dimension and truncation error accumulation | Higher-order tensors improve nonlinear fidelity, but the method becomes expensive quickly as state dimension and expansion order increase. | TBD | TBD | TBD |
| **Differential Algebra (DA)** | Automated high-order Taylor expansion of dynamics | Local and truncated polynomial representation | Loss of validity outside local expansion region | DA can propagate high-order nonlinear effects efficiently, but the approximation remains local and can lose accuracy when the uncertainty region becomes large. | TBD | TBD | TBD |
| **Multi-fidelity (MF) propagation** | Reduces cost via surrogate correction | Depends on the quality of the low-fidelity model | Model mismatch leading to biased correction updates | If the low-fidelity dynamics do not capture the dominant structure of the true motion, the correction strategy can become inaccurate or inefficient. | TBD | TBD | TBD |
| **Gaussian Mixture Models (GMMs)** | Flexible representation of multimodal distributions | Component management is difficult | Component explosion or poor tail representation due to improper splitting/merging | GMMs can represent complex distributions, but they require careful splitting, merging, and weighting to avoid excessive cost or poor tail approximation. | TBD | TBD | TBD |
| **GMM–UT / GMM–STT / GMM–DA / GMM–PCE** | Combines local Gaussian decomposition with nonlinear propagation | Inherit limitations of both mixture and propagator | Compounded approximation error and mixture proliferation | These hybrid methods are more flexible, but they still suffer from component proliferation, local approximation error, or dimensionality growth. | TBD | TBD | TBD |
| **AEGIS** | Adaptive entropy-based refinement of distribution | Adaptive splitting can become computationally heavy | Rapid growth in number of Gaussian components over time | It improves PDF representation, but repeated splitting increases the number of components and can make long-term propagation expensive. | TBD | TBD | TBD |
| **Moment-based propagation** | Very low computational cost and simplicity | Retains only low-order information | Moment closure failure under nonlinear distribution deformation | Mean and covariance alone are often insufficient once the distribution becomes curved, skewed, or multimodal. | TBD | TBD | TBD |

# Method-by-Method Discussion

## Monte Carlo

Monte Carlo simulation is the most general uncertainty propagation approach because it makes minimal assumptions about the dynamics or the distribution. Its weakness is cost. Every sample must be propagated through the full nonlinear model, so the method becomes expensive as the propagation horizon increases or as higher statistical confidence is required. This is especially restrictive in conjunction assessment, where the probability of collision may be a rare event and therefore require a very large number of samples.

## Linearised covariance

Linearised covariance propagation is efficient because it replaces the nonlinear flow with a local first-order approximation. Its accuracy is acceptable only while the uncertainty remains close to the nominal trajectory and the dynamics are weakly nonlinear. Once the distribution stretches, folds, or rotates strongly, the linearised covariance matrix no longer describes the true propagated uncertainty well. In practice, this is one of the first methods to fail when propagation time becomes long.

## Unscented transform

The UT improves on linearisation by propagating a small set of sigma points through the nonlinear dynamics. This gives a better approximation of the transformed mean and covariance than a first-order model, but it still assumes that low-order moments are enough to describe the distribution. For strongly non-Gaussian orbital uncertainty, that assumption breaks down. The UT is therefore useful as a middle-ground method, not as a universal solution.

## Polynomial chaos expansion

PCE approximates the response as an orthogonal polynomial series in the uncertain inputs. When the dimension is modest and the response is smooth, this can be highly accurate. The problem is scaling. As the number of uncertain variables or the polynomial order increases, the basis size grows rapidly. This makes the method expensive to build, harder to fit robustly, and less practical for high-dimensional orbital dynamics.

### Non-intrusive polynomial chaos

NIPC avoids modifying the governing equations directly, which makes it attractive for existing orbit propagators. The trade-off is that it still requires many propagated model evaluations to estimate the coefficients. Like standard PCE, it becomes harder to use as the uncertainty dimension increases. It also performs poorly when the input-output map is not smooth, because global polynomial surrogates are not well suited to discontinuities or sharp nonlinear transitions.

### Intrusive polynomial chaos

Intrusive PCE solves the uncertainty propagation problem by projecting the governing equations onto a polynomial basis. This can be elegant, but it is also restrictive. The original equations must be reformulated, the stochastic Galerkin system must be derived, and the implementation becomes strongly tied to the model structure. That makes the method less flexible than non-intrusive alternatives when the force model or state description changes.

## State transition tensors

STT extends the state transition matrix concept to higher orders, which makes it more capable than linear covariance methods in nonlinear regimes. The drawback is that the number of tensor terms grows quickly with expansion order and state dimension. STT is therefore powerful locally, but it is not cheap. It also requires smooth dynamics, so discontinuities, impulsive maneuvers, and other non-differentiable effects complicate its use.

## Differential algebra

DA provides an automated way to compute high-order Taylor expansions. It is often more practical than hand-derived high-order tensor methods, but it remains a local approximation. Once the uncertainty region becomes too large, the truncated polynomial map loses accuracy. DA also has to manage the growth of polynomial order and memory usage, which becomes important in realistic orbital problems.

## Multi-fidelity propagation

Multi-fidelity methods reduce cost by combining a high-fidelity model with a cheaper low-fidelity surrogate. Their performance depends on how well the low-fidelity model captures the structure of the true dynamics. If the fidelity gap is too large, the correction step can struggle. In that case, the method may lose accuracy or require more correction information than originally expected.

## GMM-based methods

GMMs are valuable because they can represent non-Gaussian and multimodal uncertainty by decomposing the distribution into Gaussian components. Their main limitation is operational complexity: the mixture must be split, merged, and propagated carefully. Too few components underfit the distribution, while too many components create excessive cost. This makes mixture management a central issue.

## Hybrid GMM methods

Hybrid methods such as GMM–UT, GMM–STT, GMM–DA, and GMM–PCE combine local Gaussian decomposition with a nonlinear propagation mechanism. They are often more flexible than a single global approximation, but they inherit the limitations of both parts. If the mixture is too coarse, the distribution is poorly represented. If the inner method is too local or too low order, nonlinear deformation is still missed.

## AEGIS

AEGIS introduces adaptive splitting based on entropy change, which helps it respond to nonlinear deformation. The limitation is that adaptive refinement can quickly increase the number of Gaussian components. Over long propagation intervals this may become expensive, and the benefit over simpler methods can diminish.

## Moment-based propagation

Moment-based methods are attractive because they are simple and computationally light. Their limitation is that they compress the full distribution into a small number of statistics. Once the propagated uncertainty becomes skewed, curved, or multimodal, the mean and covariance are no longer enough. This makes moment-based methods suitable for screening and approximate prediction, but not always for detailed collision-risk analysis.

# Overall conclusion

The limitations of orbital uncertainty propagation methods are shaped by the same underlying problem: nonlinear orbital motion transforms simple initial uncertainty into a distribution that is increasingly difficult to represent. Linear and moment-based methods are efficient but locally valid. Polynomial and tensor methods improve nonlinear fidelity but grow expensive. Monte Carlo is most general but least efficient. GMM and multi-fidelity methods sit in between, offering more flexibility at the cost of more complex model management.

For practical space-situational-awareness problems, the most effective strategy is usually not to rely on a single method, but to choose a method whose assumptions match the propagation regime, the available computation budget, and the level of distributional detail required by the mission.